In [2]:
import pandas as pd
import numpy as np
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/darshan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/darshan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [4]:
df=pd.read_csv('/Users/darshan/Innomatics/gen_ai/NLP2/IMDB Dataset.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/Users/darshan/Innomatics/gen_ai/NLP2/IMDB Dataset.csv'

In [21]:
df.columns

Index(['review', 'sentiment'], dtype='object')

In [6]:
df['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [ ]:
stop_words=set(stopwords.words('english'))
lemmatizer=WordNetLemmatizer()
def preprocess_text(text):
    text=text.lower()
    text=re.sub(r'http\S+|www\S+', '', text)
    text=text.translate(str.maketrans('', '', string.punctuation))
    text=re.sub(r'\d+', '', text)
    words=text.split()
    words=[lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return " ".join(words)
df['clean_text']=df['review'].apply(preprocess_text)

In [ ]:
df[['review','clean_text']].head()

,review,clean_text
0,One of the other reviewers has mentioned that ...,one reviewer mentioned watching oz episode you...
1,A wonderful little production. <br /><br />The...,wonderful little production br br filming tech...
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,basically there family little boy jake think t...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter matteis love time money visually stunni...


In [ ]:
bow=CountVectorizer(max_features=5000)
X_bow=bow.fit_transform(df['clean_text']).toarray()

In [ ]:
tfidf=TfidfVectorizer(max_features=5000)
X_tfidf=tfidf.fit_transform(df['clean_text']).toarray()

In [ ]:
y=df['sentiment']
X_train_bow,X_test_bow,y_train,y_test=train_test_split(X_bow,y,test_size=0.2,random_state=42)
X_train_tfidf,X_test_tfidf,_,_=train_test_split(X_tfidf,y,test_size=0.2,random_state=42)

In [ ]:
lr=LogisticRegression()
lr.fit(X_train_tfidf,y_train)
y_pred_lr=lr.predict(X_test_tfidf)

In [ ]:
nb=MultinomialNB()
nb.fit(X_train_bow,y_train)
y_pred_nb=nb.predict(X_test_bow)

In [ ]:
dt=DecisionTreeClassifier()
dt.fit(X_train_tfidf,y_train)
y_pred_dt=dt.predict(X_test_tfidf)

In [ ]:
def evaluate(y_test,y_pred,model_name):
    print(f"\n{model_name} performance:")
    print("accuracy:",accuracy_score(y_test,y_pred))
    print("precision:",precision_score(y_test,y_pred,average='weighted'))
    print("recall:",recall_score(y_test,y_pred,average='weighted'))
    print("F1 Score:",f1_score(y_test,y_pred,average='weighted'))

In [ ]:
evaluate(y_test,y_pred_lr,"Logistic Regression")
evaluate(y_test,y_pred_nb,"Naive Bayes")
evaluate(y_test,y_pred_dt,"Decision Tree")


Logistic Regression Performance:
Accuracy: 0.8846
Precision: 0.8848144493161552
Recall: 0.8846
F1 Score: 0.8845710828086292

Naive Bayes Performance:
Accuracy: 0.8465
Precision: 0.8465418714671908
Recall: 0.8465
F1 Score: 0.8465022365164928

Decision Tree Performance:
Accuracy: 0.7158
Precision: 0.7158766729718747
Recall: 0.7158
F1 Score: 0.7158012277444421
